#Transform Constructors Data
1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardize column names using snake_case (constructorId → constructor_id)
4. Rename columns to make them more meaningful (name → constructor_name )
5. Remove duplicate records
6. Transform column values of nationality to Title Case
7. Write the transformed data to silver constructors table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")  

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"


Step 1-Read bronze constructors table

In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_df = (
    spark.table(bronze_table)
         .filter((F.col("batch_id") == v_batch_id))
)

#                            (OR)
# constructors_df = spark.read.option("versionAsOf",0).table(bronze_table)

In [0]:
display(constructors_df)

Step 2- Keep only the columns required for analytics (Drop url column)

In [0]:
constructors_dropped_df = constructors_df.drop("url")


In [0]:
display(constructors_dropped_df)

Step 3 & 4:
- Standardize column names using snake_case (constructorId → constructor_id)
- Rename columns to make them more meaningful (name → constructor_name )

In [0]:
"""circuits_renamed_df = (

        circuits_selected_df
            .withColumnRenamed("circuitId","circuit_id")
            .withColumnRenamed("raceName","race_name")
            .withColumnRenamed("date","race_date") 
                                                    )

                      OR                             """
constructors_renamed_df = (constructors_dropped_df
                       .withColumnsRenamed({
                           "constructorId":"constructor_id",
                           "name":"constructor_name"
                           })
                       )


display(constructors_renamed_df)




Step 5- Remove duplicate records

In [0]:
"""constructors_distinct_df = constructors_renamed_df.distinct()
                        OR  """
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])
constructors_distinct_df.display()

Step 6- Transform column values of nationality to Title Case

In [0]:
constructors_final_df = (constructors_distinct_df
                            .withColumn("nationality",F.initcap(F.col("nationality")))

)
constructors_final_df.display()



Step 8- Write the transformed data to silver constructors table

In [0]:
write_to_silver(
    input_df=constructors_final_df,
    target_table=silver_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))